In [14]:
import joblib

saved_model = joblib.load("models/random_forest_offtask.joblib")

print("Model loaded successfully!")
print("Model type:", type(saved_model["model"]).__name__)
print("Features:", saved_model["features"])

Model loaded successfully!
Model type: RandomForestRegressor
Features: ['timetaken', 'attemptcount', 'correct', 'hintcount', 'consecutiveerrorsinrow']


In [13]:
import joblib
from pathlib import Path

# Create models folder if it doesn't exist
models_folder = Path("models")
models_folder.mkdir(exist_ok=True)

# Save the trained Random Forest model
model_path = models_folder / "random_forest_offtask.joblib"

joblib.dump({
    "model": rf_model,
    "features": X_train.columns.tolist(),
    "target": "res_offtask",
    "model_type": "experimental_baseline"
}, model_path)

print("Model saved successfully!")
print("Location:", model_path)

Model saved successfully!
Location: models\random_forest_offtask.joblib


In [12]:
# Train using only response time
X_train_time = X_train[["timetaken"]]
X_test_time = X_test[["timetaken"]]

rf_time_only = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_time_only.fit(X_train_time, y_train)

pred_time_only = rf_time_only.predict(X_test_time)

print("RESPONSE TIME ONLY")
print("MAE:", mean_absolute_error(y_test, pred_time_only))
print("R2:", r2_score(y_test, pred_time_only))

print("\nALL FIVE FEATURES")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

RESPONSE TIME ONLY
MAE: 0.07191066831326381
R2: 0.5126310049534175

ALL FIVE FEATURES
MAE: 0.06931563314146119
R2: 0.5291630160286954


In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Remove response time
features_without_time = [
    "attemptcount",
    "correct",
    "hintcount",
    "consecutiveerrorsinrow"
]

# Keep the same student-based training and testing split
X_train_no_time = X_train[features_without_time]
X_test_no_time = X_test[features_without_time]

# Train a comparison model
rf_no_time = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_no_time.fit(X_train_no_time, y_train)

# Evaluate
pred_no_time = rf_no_time.predict(X_test_no_time)

print("MODEL WITHOUT RESPONSE TIME")
print("MAE:", mean_absolute_error(y_test, pred_no_time))
print("R2:", r2_score(y_test, pred_no_time))

print("\nORIGINAL MODEL")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MODEL WITHOUT RESPONSE TIME
MAE: 0.12697628117355003
R2: 0.033289978953217725

ORIGINAL MODEL
MAE: 0.06931563314146119
R2: 0.5291630160286954


In [10]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("Random Forest Feature Importance:")
display(importance)

Random Forest Feature Importance:


,Feature,Importance
0,timetaken,0.955695
2,correct,0.013929
1,attemptcount,0.013751
3,hintcount,0.013373
4,consecutiveerrorsinrow,0.003252


In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Create Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

# Train the model
print("Training Random Forest model...")

rf_model.fit(X_train, y_train)

print("Model training completed!")

# Make predictions
y_pred = rf_model.predict(X_test)

# Evaluate model
print("\nModel Evaluation Results:")
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

Training Random Forest model...
Model training completed!

Model Evaluation Results:
Mean Absolute Error: 0.06931563314146119
R2 Score: 0.5291630160286954


In [8]:
from sklearn.model_selection import GroupShuffleSplit

# Input gameplay features
X = df[features].copy()

# Target: existing off-task score
y = df["res_offtask"].copy()

# Student IDs
groups = df["itest_id"]

# Split by student, not by individual gameplay record
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("Training students:", groups.iloc[train_idx].nunique())
print("Testing students:", groups.iloc[test_idx].nunique())

print(
    "Students appearing in both groups:",
    len(
        set(groups.iloc[train_idx])
        & set(groups.iloc[test_idx])
    )
)

Training records: 747646
Testing records: 195170
Training students: 1367
Testing students: 342
Students appearing in both groups: 0


In [7]:
print("Number of unique students:")
print(df["itest_id"].nunique())

print("\nOff-task score statistics:")
print(df["res_offtask"].describe())

print("\nOff-task score distribution:")
print(df["res_offtask"].value_counts().head(15))

print("\nGameplay feature statistics:")
display(df[features].describe().T)

Number of unique students:
1709

Off-task score statistics:
count    942816.000000
mean          0.172212
std           0.216997
min           0.000000
25%           0.048295
50%           0.122595
75%           0.122595
max           1.000000
Name: res_offtask, dtype: float64

Off-task score distribution:
res_offtask
0.122595    517899
0.048295    275205
0.889219     20984
0.118054     19893
0.440265     16556
0.889219     15684
0.468252     13777
0.916914     10686
0.785585      8893
0.108417      8477
0.785585      6126
0.149121      5140
0.734126      4150
0.833829      4051
0.778438      2463
Name: count, dtype: int64

Gameplay feature statistics:


,count,mean,std,min,25%,50%,75%,max
timetaken,942816.0,29.747869,72.019768,0.0,5.0,11.0,30.0,9999.0
attemptcount,942816.0,2.673605,2.929801,1.0,1.0,2.0,3.0,91.0
correct,942816.0,0.372681,0.483519,0.0,0.0,0.0,1.0,1.0
hintcount,942816.0,1.218490,1.980665,0.0,0.0,0.0,2.0,56.0
consecutiveerrorsinrow,942816.0,0.155307,0.885692,0.0,0.0,0.0,0.0,56.0


In [6]:
# Gameplay features we want to investigate
features = [
    "timetaken",
    "attemptcount",
    "correct",
    "hintcount",
    "consecutiveerrorsinrow"
]

print("Available gameplay features:\n")

for feature in features:
    if feature in df.columns:
        print(f"{feature}: Available")
    else:
        print(f"{feature}: Not found")

print("\nSample gameplay records:")

available = [f for f in features if f in df.columns]

display(df[available].head(10))

print("\nMissing values:")
print(df[available].isnull().sum())

Available gameplay features:

timetaken: Available
attemptcount: Available
correct: Available
hintcount: Available
consecutiveerrorsinrow: Available

Sample gameplay records:


,timetaken,attemptcount,correct,hintcount,consecutiveerrorsinrow
0,49.0,1,0,1,0
1,4.0,1,1,0,0
2,6.0,1,0,0,0
3,18.0,2,0,0,1
4,2.0,1,1,0,0
5,59.0,2,0,1,0
6,15.0,3,1,1,0
7,61.0,1,0,0,0
8,18.0,1,0,0,0
9,0.0,2,0,0,0



Missing values:
timetaken                 0
attemptcount              0
correct                   0
hintcount                 0
consecutiveerrorsinrow    0
dtype: int64


In [4]:
# Find all RES behaviour-related columns
res_columns = [col for col in df.columns if col.startswith("res_")]

print("Behaviour-related columns:")
print(res_columns)

print("\nSample values:")
display(df[res_columns].head(10))

print("\nValue ranges and missing values:")
display(df[res_columns].describe().T)

print("\nMissing values:")
print(df[res_columns].isnull().sum())

Behaviour-related columns:
['res_bored', 'res_concentrating', 'res_confused', 'res_frustrated', 'res_offtask', 'res_gaming']

Sample values:


,res_bored,res_concentrating,res_confused,res_frustrated,res_offtask,res_gaming
0,0.376427,0.320317,0.000000,0.0,0.785585,0.000264
1,0.156027,0.995053,0.887452,0.0,0.468252,0.001483
2,0.156027,0.995053,0.887452,0.0,0.468252,0.001483
3,0.156027,0.744520,0.000000,0.0,0.108417,0.010665
4,0.156027,0.744520,0.000000,0.0,0.108417,0.010665
5,0.376427,0.587839,0.000000,0.0,0.785585,0.002026
6,0.156027,0.788761,0.000000,1.0,0.108417,0.005952
7,0.376427,0.511752,0.060808,0.0,0.785585,0.010665
8,0.156027,0.683401,0.060808,0.0,0.916914,0.012562
9,0.156027,0.683401,0.060808,0.0,0.916914,0.012562



Value ranges and missing values:


,count,mean,std,min,25%,50%,75%,max
res_bored,942816.0,0.232949,0.116371,1.560266e-01,0.156027,0.156027,0.376427,0.505313
res_concentrating,942816.0,0.658442,0.173428,8.890000e-07,0.511752,0.711548,0.772610,1.000000
res_confused,942816.0,0.098940,0.249505,0.000000e+00,0.000000,0.000000,0.000000,1.000000
res_frustrated,942816.0,0.131406,0.300351,0.000000e+00,0.000000,0.000000,0.009561,1.000000
res_offtask,942816.0,0.172212,0.216997,0.000000e+00,0.048295,0.122595,0.122595,1.000000
res_gaming,942816.0,0.192703,0.340232,1.210000e-06,0.001483,0.005797,0.259648,0.999377



Missing values:
res_bored            0
res_concentrating    0
res_confused         0
res_frustrated       0
res_offtask          0
res_gaming           0
dtype: int64


In [3]:
# Display all available columns
print("ALL DATASET COLUMNS:\n")
for col in df.columns:
    print(col)

# Find possible behaviour-related columns
print("\nBEHAVIOUR-RELATED COLUMNS:\n")

keywords = [
    "res_", "time", "attempt", "hint",
    "error", "correct", "click", "action"
]

behaviour_columns = [
    col for col in df.columns
    if any(word in col for word in keywords)
]

print(behaviour_columns)

# Check missing values
print("\nMISSING VALUES:\n")
print(df[behaviour_columns].isnull().sum())

ALL DATASET COLUMNS:

itest_id
sy assistments usage
aveknow
avecarelessness
avecorrect
numactions
averesbored
averesengcon
averesconf
averesfrust
averesofftask
averesgaming
actionid
skill
problemid
assignmentid
assistmentid
starttime
endtime
timetaken
correct
original
hint
hintcount
hinttotal
scaffold
bottomhint
attemptcount
problemtype
frishelprequest
frpast5helprequest
frpast8helprequest
stlhintused
past8bottomout
totalfrpercentpastwrong
totalfrpastwrongcount
frpast5wrongcount
frpast8wrongcount
totalfrtimeonskill
timesinceskill
frworkinginschool
totalfrattempted
totalfrskillopportunities
responseisfillin
responseischosen
endswithscaffolding
endswithautoscaffolding
frtimetakenonscaffolding
frtotalskillopportunitiesscaffolding
totalfrskillopportunitiesbyscaffolding
frishelprequestscaffolding
timegreater5secprev2wrong
sumright
helpaccessunder2sec
timegreater10secandnextactionright
consecutiveerrorsinrow
sumtime3sdwhen3rowright
sumtimeperskill
totaltimebypercentcorrectforskill
prev5count

In [2]:
import pandas as pd
from pathlib import Path

data_folder = Path("data")

# Load all 10 student log files
datasets = []

for i in range(1, 11):
    file_path = data_folder / f"student_log_{i}.csv"
    df_part = pd.read_csv(file_path, low_memory=False)

    # Standardize column names
    df_part.columns = df_part.columns.str.strip().str.lower()

    datasets.append(df_part)
    print(f"Student Log {i}: {len(df_part)} records")

# Combine into one dataset
df = pd.concat(datasets, ignore_index=True)

print("\nTotal records:", len(df))
print("Total columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())

display(df.head())

Student Log 1: 231403 records
Student Log 2: 365839 records
Student Log 3: 37160 records
Student Log 4: 30659 records
Student Log 5: 66726 records
Student Log 6: 63553 records
Student Log 7: 58002 records
Student Log 8: 10542 records
Student Log 9: 58294 records
Student Log 10: 20638 records

Total records: 942816
Total columns: 76
Duplicate rows: 0


,itest_id,sy assistments usage,aveknow,avecarelessness,avecorrect,numactions,averesbored,averesengcon,averesconf,averesfrust,...,confidence(off task),confidence(gaming),res_bored,res_concentrating,res_confused,res_frustrated,res_offtask,res_gaming,ln-1,ln
0,8,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,0.115905,0.112408,...,0.838710,0.008522,0.376427,0.320317,0.000000,0.0,0.785585,0.000264,0.13,0.06119
1,8,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,0.115905,0.112408,...,0.600000,0.047821,0.156027,0.995053,0.887452,0.0,0.468252,0.001483,0.06119,0.21351
2,8,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,0.115905,0.112408,...,0.600000,0.047821,0.156027,0.995053,0.887452,0.0,0.468252,0.001483,0.116,0.033306
3,8,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,0.115905,0.112408,...,0.204082,0.343996,0.156027,0.744520,0.000000,0.0,0.108417,0.010665,0.116,0.033306
4,8,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,0.115905,0.112408,...,0.204082,0.343996,0.156027,0.744520,0.000000,0.0,0.108417,0.010665,0.033306,0.118386


In [1]:
print("MathQuest Machine Learning Environment Ready!")


MathQuest Machine Learning Environment Ready!
